# YouTube コメント分析（汎用）

旧09ベース。動画IDを変えるだけで任意の動画のコメントを分析できる。

出力: コメントTSV / 単語頻度CSV / 頻度バーチャート / ワードクラウドPNG

語彙（1語として扱う複合語・除外語・類義語）は `config.VOCAB_PRESETS` のプリセット +
このノートブック上での追加で調整する。

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
!pip install -q nagisa wordcloud japanize-matplotlib
drive.mount("/content/drive")

In [ ]:
#@title ⚙️ 設定
#@markdown 分析したい動画IDをカンマ区切りで入力
VIDEO_IDS_STR = "v4caB015k48, lSwfNemmzX8"  #@param {type:"string"}
#@markdown 取得コメントの最大件数（0 で無制限）
MAX_COMMENTS = 0  #@param {type:"integer"}
#@markdown 頻度バーチャートの表示件数
TOP_N = 30  #@param {type:"integer"}
#@markdown 語彙プリセット
PRESET = "default"  #@param ["default", "osomatsu"]
#@markdown 追加の複合語（カンマ区切り・空可）
EXTRA_SINGLE_WORDS = ""  #@param {type:"string"}
#@markdown 追加の除外語（カンマ区切り・空可）
EXTRA_STOP_WORDS = ""  #@param {type:"string"}
#@markdown ワードクラウドの固定配色（カンマ区切りHEX・空でランダム）
WORDCLOUD_COLOR_PALETTE = "#87CEEB,#FF69B4,#A9A9A9,#43528F,#F0E68C"  #@param {type:"string"}
#@markdown マスク画像パス（空なら長方形）
MASK_IMAGE_PATH = ""  #@param {type:"string"}

VIDEO_IDS = [v.strip() for v in VIDEO_IDS_STR.split(",") if v.strip()]

In [ ]:
#@title 📥 コメント取得 + テキスト整形
from sixfonia_analytics import auth, comment

youtube = auth.build_youtube()

all_video_dfs = {}
for vid in VIDEO_IDS:
    df_vid = comment.fetch_comments(youtube, vid, MAX_COMMENTS)
    if not df_vid.empty:
        df_vid["text"] = comment.clean_text(df_vid["text"])
        all_video_dfs[vid] = df_vid

In [ ]:
#@title 🈁 単語抽出と頻度集計
from sixfonia_analytics.comment import WordExtractor, word_freq_df
from sixfonia_analytics.config import VOCAB_PRESETS

preset = VOCAB_PRESETS[PRESET]
extractor = WordExtractor(
    single_words=preset["single_words"]
        + [w.strip() for w in EXTRA_SINGLE_WORDS.split(",") if w.strip()],
    stop_words=set(preset["stop_words"])
        | {w.strip() for w in EXTRA_STOP_WORDS.split(",") if w.strip()},
    synonym_map=preset["synonym_map"],
)

all_video_word_counts = {}
for vid, df_cur in all_video_dfs.items():
    counts = extractor.count(df_cur["text"])
    all_video_word_counts[vid] = counts
    word_freq_df(counts).to_csv(f"word_freq_{vid}.csv", index=False)
    print(f"--- {vid} TOP20 ---")
    print(counts.most_common(20))

In [ ]:
#@title 📊 頻度バーチャート
import matplotlib.pyplot as plt
from sixfonia_analytics import plots

plots.setup_japanese_font()

for vid, counts in all_video_word_counts.items():
    if not counts:
        continue
    words, nums = zip(*counts.most_common(TOP_N))
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(words)), nums, color="#388E3C")
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_xlabel("出現回数")
    ax.set_title(f"コメント単語頻度 TOP{TOP_N} — {vid}")
    plt.tight_layout()
    plt.show()

In [ ]:
#@title ☁️ ワードクラウド（固定配色 / マスク対応）
import os
import random
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from PIL import Image
from wordcloud import WordCloud

font_path = fm.findfont("IPAexGothic")
palette = [c.strip() for c in WORDCLOUD_COLOR_PALETTE.split(",") if c.strip()] or ["#333333"]

def load_mask(path, max_size=1200):
    if not path or not os.path.exists(path):
        return None
    img = Image.open(path)
    if img.mode in ("RGBA", "LA"):
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[-1])
        img = bg
    else:
        img = img.convert("RGB")
    if max(img.size) > max_size:
        ratio = max_size / max(img.size)
        img = img.resize((int(img.size[0] * ratio), int(img.size[1] * ratio)), Image.LANCZOS)
    return np.array(img)

def rank_color_func(counts):
    order = {w: i for i, (w, _) in enumerate(counts.most_common())}
    def _f(word, **kwargs):
        return palette[order.get(word, 0) % len(palette)]
    return _f

mask = load_mask(MASK_IMAGE_PATH)
for vid, counts in all_video_word_counts.items():
    if not counts:
        continue
    wc = WordCloud(
        font_path=font_path, background_color="white", max_words=200,
        color_func=rank_color_func(counts), random_state=42,
        mask=mask, width=800, height=600, collocations=False,
    ).generate_from_frequencies(counts)
    plt.figure(figsize=(10, 8))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud — {vid}")
    plt.show()
    wc.to_file(f"wordcloud_{vid}.png")
    print(f"保存: wordcloud_{vid}.png")

In [ ]:
#@title 📈 キーワード時系列（空ならスキップ）
KEYWORDS = ""  #@param {type:"string"}

import matplotlib.pyplot as plt
from sixfonia_analytics import comment

keywords = [k.strip() for k in KEYWORDS.split(",") if k.strip()]
if keywords:
    for vid, df_cur in all_video_dfs.items():
        daily = comment.keyword_daily_counts(df_cur, keywords)
        if daily.empty:
            print(f"{vid}: キーワードなし")
            continue
        display(daily)
        daily.plot(kind="bar", figsize=(10, 5), title=f"日別キーワード言及数 — {vid}")
        plt.tight_layout()
        plt.show()
else:
    print("KEYWORDS が空のためスキップ")

In [ ]:
#@title 💾 エクスポート（コメントTSV + 頻度CSV + ワードクラウドPNG を zip でダウンロード）
import glob
import os
import zipfile
from google.colab import files

for vid, df_cur in all_video_dfs.items():
    df_cur.to_csv(f"comments_{vid}.tsv", sep="\t", index=False)

zip_name = "comment_analysis_results.zip"
with zipfile.ZipFile(zip_name, "w") as zf:
    for vid in VIDEO_IDS:
        for f in glob.glob(f"*{vid}*"):
            if os.path.isfile(f) and f != zip_name:
                zf.write(f)

files.download(zip_name)